# Phase 5 — Disease Module Training (Colab)

Three-stage cascade for the IKS-grounded multimodal agricultural advisory
system at IIITDM Jabalpur (Author: Ankit Pawar, Supervisor: Dr. Akshay
Pandey).

**Stages:**
1. **Pretrain on PlantVillage** (38 classes, 25 epochs)
2. **Fine-tune on Paddy Doctor** (10 classes, 20 epochs)
3. **Fine-tune on PlantDoc** (27 classes, 30 epochs)

Each stage is **independently resumable** — its checkpoint is pushed to a
private HF Hub model repo after every epoch. If the Colab session resets,
re-run the same cell with the `--resume` flag and training continues from
the latest checkpoint.

## ⚠️ Before you start

- Set the runtime to **GPU** (Runtime → Change runtime type → T4 GPU).
- **Colab free tier limits:** ~12 h continuous session, ~3–4 h daily GPU
  quota. Plan to run one stage per session if you're on free tier;
  Colab Pro / Pro+ raise the quota substantially.
- The notebook reads datasets from private HF Hub repos. You will be
  prompted to log in with a HF Hub Write token (Cell 3).


In [ ]:
# Cell 2 — setup
# If your repo URL differs from the line below, edit just this one line:
REPO_URL = "https://github.com/ankit8453/iks-rag-thesis.git"
REPO_PATH = "/content/iks-rag-thesis"

import os
import subprocess
if not os.path.isdir(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
os.chdir(REPO_PATH)

# Install the project's runtime dependencies. Colab pre-installs
# torch / numpy / pandas / matplotlib so we skip those.
_pip_packages = [
    "timm>=1.0.0",
    "albumentations>=1.4",
    "huggingface_hub>=0.24",
    "datasets>=2.20",
    "pytorch-grad-cam>=1.5",
    "iterative-stratification>=0.1.7",
    "pydantic>=2.7",
]
subprocess.run(["pip", "install", "--quiet", *_pip_packages], check=True)
print("setup ok")

In [ ]:
# Cell 3 — HF Hub login
from huggingface_hub import notebook_login
notebook_login()  # paste your Write token when prompted

In [ ]:
# Cell 4 — verify GPU + pick batch size
import subprocess, torch
subprocess.run(["nvidia-smi"], check=False)
print()
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f"GPU: {dev.name}, VRAM: {dev.total_memory / 1024**3:.1f} GiB")
else:
    print("No GPU detected — switch runtime to GPU before continuing.")

import sys
sys.path.insert(0, REPO_PATH)
from src.disease.train import auto_batch_size
print("auto_batch_size for B4 at 380x380:", auto_batch_size(380))

## Stage 1 — Pretrain on PlantVillage

**25 epochs.** Pushes checkpoints to `ankit-iiitdmj/iks-disease-plantvillage`.
Re-running this cell with `--resume` picks up from the latest checkpoint if
the previous session got interrupted.

In [ ]:
# Cell 6 — Stage 1: PlantVillage pretraining
!python -m src.disease.train --stage pretrain --resume

## Stage 2 — Fine-tune on Paddy Doctor

**20 epochs.** Seeds from the final PlantVillage checkpoint automatically.
Only run this once Stage 1 reaches max epochs (or you'll be fine-tuning a
half-trained model).

In [ ]:
# Cell 8 — Stage 2: Paddy Doctor fine-tune
!python -m src.disease.train --stage finetune_paddy --resume

## Stage 3 — Fine-tune on PlantDoc

**30 epochs.** Seeds from the final Paddy Doctor checkpoint. This is the
real-field-imagery stage — performance here is what gets reported in the
thesis.

In [ ]:
# Cell 10 — Stage 3: PlantDoc fine-tune
!python -m src.disease.train --stage finetune_plantdoc --resume

## Evaluation on held-out test sets

Run final eval on each dataset's test split. Prints per-class
precision/recall/F1 + confusion matrix. The JSON gets pushed to HF Hub
alongside the final model checkpoint.

In [ ]:
# Cell 12 — Test-set evaluation
import json
import torch
from huggingface_hub import HfApi

from src.disease.train import (
    STAGE_INFO,
    CheckpointManager,
    auto_batch_size,
    _build_loaders_from_hf,
    _build_model_for_stage,
    evaluate,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
batch = auto_batch_size()
api = HfApi()

for stage in ("pretrain", "finetune_paddy", "finetune_plantdoc"):
    info = STAGE_INFO[stage]
    # Re-build the test loader for this dataset.
    train_loader, val_loader, num_classes = _build_loaders_from_hf(
        stage, batch_size=batch, num_workers=2
    )
    # Pull the best checkpoint.
    ckpt_mgr = CheckpointManager(info["model_repo"])
    ckpt = ckpt_mgr.try_load_latest()
    if ckpt is None:
        print(f"Skip {stage}: no checkpoint on HF Hub.")
        continue
    model = _build_model_for_stage(stage, num_classes, ckpt)
    model.load_state_dict(ckpt["model_state"], strict=False)
    model.to(device).eval()

    metrics = evaluate(model, val_loader, num_classes, device)
    summary = metrics.to_dict()
    print(f"\n=== {stage} eval ===")
    print(f"  top1 acc: {summary['top1_accuracy']:.4f}")
    print(f"  top5 acc: {summary['top5_accuracy']:.4f}")
    print(f"  macro F1: {summary['macro_f1']:.4f}")

    metrics_path = f"/tmp/{stage}_metrics.json"
    with open(metrics_path, "w") as fh:
        json.dump(summary, fh, indent=2)
    api.upload_file(
        path_or_fileobj=metrics_path,
        path_in_repo="eval_metrics.json",
        repo_id=info["model_repo"],
        repo_type="model",
    )
    print(f"  pushed eval metrics -> {info['model_repo']}/eval_metrics.json")

## What's next

After all three stages complete and the eval metrics are pushed:

1. From your laptop, pull the final PlantDoc checkpoint back via
   `huggingface-cli download ankit-iiitdmj/iks-disease-plantdoc
   checkpoint_best.pt --local-dir models/`.
2. Move on to **Phase 6** (soil module training) using
   `notebooks/phase6_soil_training.ipynb` (separate prompt).
3. Phase 8 integrates the trained disease module into the multimodal
   context constructor.
